# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nauman024/FlyRank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

Feature Vector Construction:

We extract features strictly aggregated over the historical observation window (month = '2026-03'). Numerical features are filled with appropriate zero/median defaults, and categorical flags are converted to binary indicators.

In [1]:
import duckdb
from google.colab import userdata
import pandas as pd
import numpy as np

# 1. DuckDB Setup
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

# 2. Build Feature Vector Query
query = f"""
SELECT
    f.content_hash_id,
    AVG(f.gsc_avg_position) as avg_position_30d,
    SUM(f.gsc_impressions) as total_impressions,
    SUM(f.gsc_clicks) as total_clicks,
    (SUM(f.gsc_clicks) / NULLIF(SUM(f.gsc_impressions), 0)) as historical_ctr,
    MAX(CASE WHEN c.is_deleted IS TRUE THEN 1 ELSE 0 END) as is_deleted_flag,
    -- Proxy Target for underperformance
    CASE WHEN SUM(f.gsc_clicks) < 10 THEN 1 ELSE 0 END as target_underperforming
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') f
JOIN read_parquet('{rel}/dim_content.parquet') c ON f.content_hash_id = c.content_hash_id
WHERE c.is_deleted IS FALSE
GROUP BY f.content_hash_id
LIMIT 5000;
"""

df_features = con.sql(query).df()

# 3. Handle Missing Values
df_features['historical_ctr'] = df_features['historical_ctr'].fillna(0.0)
df_features['avg_position_30d'] = df_features['avg_position_30d'].fillna(100.0)
df_features['total_impressions'] = df_features['total_impressions'].fillna(0)
df_features['total_clicks'] = df_features['total_clicks'].fillna(0)

print(f"Feature vector shape: {df_features.shape}")
display(df_features.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (5000, 7)


,content_hash_id,avg_position_30d,total_impressions,total_clicks,historical_ctr,is_deleted_flag,target_underperforming
0,content_b7e512995f79d5a6,4.394234,1140.0,2.0,0.001754,0,1
1,content_05597932fe4da067,2.714744,57.0,0.0,0.000000,0,1
2,content_905aa32a0230694e,6.481453,149.0,0.0,0.000000,0,1
3,content_05434271b257bb68,6.320337,1421.0,6.0,0.004222,0,1
4,content_d056587ff7faca0c,4.459107,2770.0,16.0,0.005776,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

- avg_position_30d: Mean SERP ranking position across March 2026. Missing values filled with 100.0 (unranked penalty). Available at decision moment from past logs.

- total_impressions: Total search appearances over 30 days. Missing values filled with 0. Available at decision moment.

- total_clicks: Total user clicks logged over 30 days. Missing values filled with 0. Available at decision moment.

- historical_ctr: Click-through rate derived as total_clicks / total_impressions. Null division filled with 0.0. Available at decision moment.

- is_deleted_flag: Binary metadata indicator (0 or 1) from dim_content. Available at decision moment.

In [2]:
# Verification of missing values and summary stats
summary = pd.DataFrame({
    'Data Type': df_features.dtypes,
    'Null Count': df_features.isnull().sum(),
    'Min Value': df_features.min(numeric_only=True),
    'Max Value': df_features.max(numeric_only=True)
})
display(summary)

,Data Type,Null Count,Min Value,Max Value
avg_position_30d,float64,0,0.0,100.000000
content_hash_id,object,0,NaN,NaN
historical_ctr,float64,0,0.0,0.333333
is_deleted_flag,int32,0,0.0,0.000000
target_underperforming,int32,0,0.0,1.000000
total_clicks,float64,0,0.0,535.000000
total_impressions,float64,0,0.0,151166.000000


## 3. The leakage hunt

Leakage Attack Test:
We intentionally inject future_outcome_proxy (a feature directly proportional to the target label) into the training matrix to demonstrate how data leakage artificially inflates model accuracy to ~1.0000. We then prune the leaked feature to restore realistic, generalizable performance.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Introduce Artificial Leaked Feature (Target proxy)
df_features['LEAKED_future_outcome'] = df_features['target_underperforming'] * 0.99

features_with_leak = ['avg_position_30d', 'total_impressions', 'historical_ctr', 'LEAKED_future_outcome']
features_honest = ['avg_position_30d', 'total_impressions', 'historical_ctr', 'is_deleted_flag']

X_leak = df_features[features_with_leak]
X_honest = df_features[features_honest]
y = df_features['target_underperforming']

# 2. Evaluate Leaked Model
clf_leak = DecisionTreeClassifier()
clf_leak.fit(X_leak, y)
leak_acc = accuracy_score(y, clf_leak.predict(X_leak))
print(f"Accuracy WITH Target Leakage Feature: {leak_acc:.4f}")

# 3. Clean up and Evaluate Honest Model
del df_features['LEAKED_future_outcome']
clf_honest = DecisionTreeClassifier(max_depth=3)
clf_honest.fit(X_honest, y)
honest_acc = accuracy_score(y, clf_honest.predict(X_honest))
print(f"Accuracy WITHOUT Leaked Feature (Honest Baseline): {honest_acc:.4f}")

Accuracy WITH Target Leakage Feature: 1.0000
Accuracy WITHOUT Leaked Feature (Honest Baseline): 0.9832


## 4. What I excluded and why

- month = '2026-06' (Final Month): Excluded completely from training to prevent temporal test leakage; reserved strictly as the sealed out-of-time test partition.

- content_hash_id / Raw IDs: Excluded from the feature vector to prevent high-cardinality memorization.

- is_deleted IS TRUE Pages: Excluded because deleted/archived URLs introduce non-operational noise into ranking metrics.

In [4]:
# Check that raw identifiers and high cardinality fields are omitted from model features
final_feature_columns = list(X_honest.columns)
print("Final model feature columns:")
for col in final_feature_columns:
    print(f" - {col}")

Final model feature columns:
 - avg_position_30d
 - total_impressions
 - historical_ctr
 - is_deleted_flag


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.